# Gizli Dirichlet Dağılımı (Latent Dirichlet Allocation) (LDA)

🎯 Bu challenge'ın amacı, **LDA** algoritması (NLP'de Denetimsiz Öğrenme) ile e-posta külliyatı içinde konular bulmaktır.

✉️ İşte 1000'den fazla ***etiketlenmemiş e-posta*** içeren bir koleksiyon. Bunlardan ***konuları çıkarmaya*** çalışalım!

In [1]:
import pandas as pd

url = 'https://d32aokrjazspmn.cloudfront.net/materials/lda_data'

data = pd.read_csv(url, sep=",", header=None)
data.columns = ['text']
data.head()

,text
0,From: gld@cunixb.cc.columbia.edu (Gary L Dare)...
1,From: atterlep@vela.acs.oakland.edu (Cardinal ...
2,From: miner@kuhub.cc.ukans.edu\nSubject: Re: A...
3,From: atterlep@vela.acs.oakland.edu (Cardinal ...
4,From: vzhivov@superior.carleton.ca (Vladimir Z...


In [2]:
data.shape

(1199, 1)

## (1) Preprocessing 

❓ **Question (Cleaning**) ❓ You're used to it by now... Clean up! Store the cleaned text in a new column "clean_text" of the DataFrame.

In [3]:
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import nltk

# Gerekli bileşenleri indirelim
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')

def clean_text(text):
    # 1. Küçük harfe çevir
    text = text.lower()
    
    # 2. Rakamları temizle
    text = ''.join(char for char in text if not char.isdigit())
    
    # 3. Noktalama işaretlerini temizle
    for punctuation in string.punctuation:
        text = text.replace(punctuation, '')
        
    # 4. Tokenize et (Kelimelere ayır)
    tokens = word_tokenize(text)
    
    # 5. Stopwords (gereksiz kelimeler) temizliği
    stop_words = set(stopwords.words('english'))
    tokens = [w for w in tokens if not w in stop_words]
    
    # 6. Lemmatization (Köklerine indirgeme - örn: "studying" -> "study")
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    return " ".join(tokens)

# Temizlenmiş metni yeni bir sütuna kaydedelim
data['clean_text'] = data['text'].apply(clean_text)

# Sonucu kontrol edelim
data.head()

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/bariscan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/bariscan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/bariscan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/bariscan/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


,text,clean_text
0,From: gld@cunixb.cc.columbia.edu (Gary L Dare)...,gldcunixbcccolumbiaedu gary l dare subject sta...
1,From: atterlep@vela.acs.oakland.edu (Cardinal ...,atterlepvelaacsoaklandedu cardinal ximenez sub...
2,From: miner@kuhub.cc.ukans.edu\nSubject: Re: A...,minerkuhubccukansedu subject ancient book orga...
3,From: atterlep@vela.acs.oakland.edu (Cardinal ...,atterlepvelaacsoaklandedu cardinal ximenez sub...
4,From: vzhivov@superior.carleton.ca (Vladimir Z...,vzhivovsuperiorcarletonca vladimir zhivov subj...


## (2) Latent Dirichlet Allocation model

❓ **Soru (Eğitim)** ❓ Potansiyel konuları çıkarmak için bir LDA modeli eğitin

In [4]:
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer

# 1. Metinleri Sayısal Vektörlere Dönüştürme (Bag of Words)
# LDA genellikle ham sayımlarla (CountVectorizer) daha iyi çalışır.
vectorizer = CountVectorizer()
data_vectorized = vectorizer.fit_transform(data['clean_text'])

# 2. LDA Modelini Oluşturma
# n_components: Çıkarmak istediğimiz konu sayısı (başlangıç için 2 idealdir)
lda_model = LatentDirichletAllocation(n_components=2, random_state=42)

# 3. Modeli Eğitme
lda_model.fit(data_vectorized)

LatentDirichletAllocation(n_components=2, random_state=42)

##  (3) Potansiyel konuları görselleştirin

🎁 Potansiyel konularla ilişkili kelimeleri yazdırmak için bir  fonksiyon kodladık.

In [5]:
def print_topics(model, vectorizer):
    for idx, topic in enumerate(model.components_):
        print("Topic %d:" % (idx))
        print([(vectorizer.get_feature_names_out()[i], topic[i])
                        for i in topic.argsort()[:-10 - 1:-1]])

❓ **Soru** ❓ LDA tarafından çıkarılan konuları yazdırın.

In [6]:
# Eğitilen model ve vectorizer'ı kullanarak konuları yazdırıyoruz
print_topics(lda_model, vectorizer)

Topic 0:
[('team', 954.492583614096), ('game', 926.6763857533701), ('line', 705.8911900292647), ('subject', 631.5763931401137), ('organization', 609.7109699597447), ('hockey', 594.4924889314436), ('player', 524.466852174314), ('play', 495.7560584698419), ('year', 443.08653264344315), ('university', 421.04582301590835)]
Topic 1:
[('god', 1490.4722687452693), ('one', 838.1900693896296), ('would', 809.9103911151723), ('christian', 758.4705977219897), ('people', 700.3520982523543), ('subject', 675.423606859835), ('line', 638.1088099706847), ('jesus', 623.4900913689678), ('organization', 561.2890300402074), ('say', 540.5486730510962)]


## (4) Yeni bir metnin belge-konu karışımını tahmin edin

❓ **Soru (Tahmin)** ❓

LDA modeliniz fit edildiğine göre, onu yeni bir metnin konularını tahmin etmek için kullanabilirsiniz.

1. Örneği vektörleştirin
2. Vektörleştirilmiş örnek üzerinde LDA'yı kullanarak konuları tahmin edin

In [8]:
example = ["My team performed poorly last season. Their best player was out injured and only played one game"]

In [10]:
# 1. Örneği vektörleştirelim
# DİKKAT: Yeni veriyi sadece 'transform' ediyoruz, 'fit' etmiyoruz!
example_vectorized = vectorizer.transform(example)

# 2. Vektörleştirilmiş örnek üzerinde LDA'yı kullanarak konuları tahmin edelim
prediction = lda_model.transform(example_vectorized)

# Tahmin sonucunu yazdıralım
print("Metin:", example[0])
print("Konu Dağılımı (Topic 0, Topic 1):", prediction)

Metin: My team performed poorly last season. Their best player was out injured and only played one game
Konu Dağılımı (Topic 0, Topic 1): [[0.95167532 0.04832468]]


🏁 Tebrikler! LDA'yı hızlı bir şekilde nasıl uygulayacağınızı öğrendiniz.

💾 Not defterinizi `git add/commit/push` yapmayı unutmayın...

🚀 ... ve bir sonraki göreve geçin!